# LIONBOT v4 — เทรนใหม่จาก base ด้วย dataset หลักสูตร 1,910 ตัวอย่าง

เทรนจาก `scb10x/typhoon2.1-gemma3-4b` (ไม่ต่อจาก v3 เพื่อไม่ให้ติดข้อมูลที่แต่งผิด) แล้ว export GGUF เป็น `typhoon-lionbot-v4`

**ตั้ง Runtime เป็น T4 GPU ก่อนรัน**

## 1. ติดตั้ง Unsloth + เช็ค GPU

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir "git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี GPU — ไปที่ Runtime > Change runtime type > T4 GPU ก่อน")

## 2. เชื่อม Google Drive + ค้นหาเช็คพอยต์เดิม

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PREV_ADAPTER_PATH = None  # v4 เทรนใหม่จาก base ตั้งใจไม่ต่อยอด v3
print('เทรนใหม่จาก base model')

## 3. โหลดโมเดล (ต่อยอดถ้าเจอ adapter / เริ่มใหม่ถ้าไม่เจอ)

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "scb10x/typhoon2.1-gemma3-4b"
MAX_SEQ_LENGTH = 3072

if PREV_ADAPTER_PATH:
    # ต่อยอด: โหลด adapter เดิมมาเทรนต่อ
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=PREV_ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    print(f"โหลดต่อยอดจาก {PREV_ADAPTER_PATH} สำเร็จ")
else:
    # fallback: เริ่มใหม่จาก base model เดิม (เหมือน v2)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    print(f"เริ่ม LoRA ใหม่จาก {BASE_MODEL}")

## 4. อัปโหลด dataset.jsonl (1,910 ตัวอย่าง)

In [ ]:
from google.colab import files
uploaded = files.upload()  # เลือก dataset.jsonl
DATASET_FILE = list(uploaded.keys())[0]
print("ใช้ไฟล์:", DATASET_FILE)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"จำนวนตัวอย่าง: {len(dataset)}")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(dataset[0]["text"][:400])
lens=[len(tokenizer(t)['input_ids']) for t in dataset['text']]
print('max tokens',max(lens),'mean',sum(lens)/len(lens))


## 5. เทรน

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)
trainer_stats = trainer.train()

## 6. เซฟ adapter ลง Drive

In [ ]:
ADAPTER_OUT = "typhoon-lionbot-v4-adapter"
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

import shutil
DRIVE_ADAPTER_DEST = f"/content/drive/MyDrive/{ADAPTER_OUT}"
shutil.copytree(ADAPTER_OUT, DRIVE_ADAPTER_DEST, dirs_exist_ok=True)
print(f"เซฟ adapter ไว้ที่ {DRIVE_ADAPTER_DEST} แล้ว — รอบหน้า (v4) notebook นี้จะหาเจอเองอัตโนมัติ")

## 7. แปลงเป็น GGUF (q4_k_m) สำหรับ Ollama

In [ ]:
GGUF_NAME = "typhoon-lionbot-v4-gguf"
model.save_pretrained_gguf(GGUF_NAME, tokenizer, quantization_method="q4_k_m")

In [ ]:
# path ที่ Unsloth สร้างจริงบางทีต่างจากชื่อโฟลเดอร์ที่ตั้งไว้ (เจอปัญหานี้ตอนเทรน v2) — ค้นด้วย glob กันพลาด
import glob, shutil, os

gguf_files = glob.glob(f"**/*.gguf", recursive=True)
gguf_files = [f for f in gguf_files if "q4_k_m" in f.lower() or "Q4_K_M" in f]
if not gguf_files:
    gguf_files = glob.glob("**/*.gguf", recursive=True)

print("เจอไฟล์ GGUF:", gguf_files)
assert gguf_files, "ไม่เจอไฟล์ .gguf — เช็ค cell ก่อนหน้าว่า export สำเร็จหรือไม่"

SRC_GGUF = gguf_files[0]
DRIVE_GGUF_DIR = "/content/drive/MyDrive/typhoon-lionbot-v4-gguf"
os.makedirs(DRIVE_GGUF_DIR, exist_ok=True)
DEST_GGUF = os.path.join(DRIVE_GGUF_DIR, "typhoon-lionbot-v4.Q4_K_M.gguf")
shutil.copy(SRC_GGUF, DEST_GGUF)
print(f"✅ คัดลอกไปที่ {DEST_GGUF} แล้ว")